<a href="https://colab.research.google.com/github/riraonline/data_science_2026/blob/main/Pertemuan3_Richardin_Rafsanjani_220401010116.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
from scipy import stats

#Muat Data
df = pd.read_csv('housing_dirty.csv')

# Eksplorasi awal
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


# **Proses Data Cleaning**

**A. Menangani Duplikat dan Normalisasi String**

In [30]:
# 1. Hapus baris duplikat
df.drop_duplicates(inplace=True)

# 2. Normalisasi String
df['kota'] = df['kota'].str.strip().str.title() # Contoh: 'jakarta' -> 'Jakarta'
df['kondisi'] = df['kondisi'].str.strip().str.lower() # Contoh: 'Baik' -> 'baik'

print(f"Jumlah baris setelah hapus duplikat: {len(df)}")

print("Variasi nama kota setelah normalisasi:")
print(df['kota'].unique())

print("\nVariasi kondisi setelah normalisasi:")
print(df['kondisi'].unique())

Jumlah baris setelah hapus duplikat: 130
Variasi nama kota setelah normalisasi:
['Jogja' 'Medan' 'Depok' 'Ygy' 'Jakarta' 'Yogyakarta' 'Bandung' 'Surabaya'
 'Dpk' 'Sby' 'Makassar' 'Mdn' 'Semarang' 'Smg' 'Bdg' 'Mksr']

Variasi kondisi setelah normalisasi:
['baik' 'bagus' 'sedang' 'baik sekali' 'rusak' 'cukup' 'perlu renovasi'
 'jelek']


**B. Imputasi Missing Values**

In [31]:
# Imputasi numerik dengan Median
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

# Imputasi kategorik dengan Modus
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print(f"Sisa missing value di harga_juta: {df['harga_juta'].isnull().sum()}")

Sisa missing value di harga_juta: 0


**C. Penanganan Outlier (IQR Fence)**

In [32]:
# Menampilkan batas dan jumlah outlier yang terdeteksi
for col in ['harga_juta', 'luas_m2']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR

    print(f"\nKolom: {col}")
    print(f"Batas Bawah: {lower}, Batas Atas: {upper}")


Kolom: harga_juta
Batas Bawah: -422.75, Batas Atas: 1719.25

Kolom: luas_m2
Batas Bawah: -145.225, Batas Atas: 512.9749999999999


# **Akses REST API**

In [33]:
import requests
import pandas as pd

URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL, timeout=10)

if response.status_code == 200:
    print("Koneksi API Berhasil!")
    data = response.json()
    df_api = pd.DataFrame(data)
    # Menampilkan 5 baris pertama data dari API
    print(df_api.head())
else:
    print(f"Gagal mengambil data. Status code: {response.status_code}")

Koneksi API Berhasil!
   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                                             address                  phone  \
0  {'street': 'Kulas Light', 'suite': 'Apt. 556',...  1-770-736-8031 x56442   
1  {'street': 'Victor Plains', 'suite': 'Suite 87...    010-692-6593 x09125   
2  {'street': 'Douglas Extension', 'suite': 'Suit...         1-463-123-4447   
3  {'street': 'Hoeger Mall', 'suite': 'Apt. 692',...      493-170-9623 x156   
4  {'street': 'Skiles Walks', 'suite': 'Suite 351...          (254)954-1289   

         website                                            company  
0  hildegard.org  {'name': 'Romaguera-Cron

# **Validasi dan Ekspor**

In [34]:
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

# Ekspor ke CSV
df.to_csv('housing_clean.csv', index=False)
print("Dataset bersih telah disimpan sebagai 'housing_clean.csv'")

print("--- Validasi Akhir ---")
print(f"Total Missing Values: {df.isnull().sum().sum()}")
print(f"Total Duplikat: {df.duplicated().sum()}")
print(f"Shape Akhir: {df.shape}")

Dataset bersih telah disimpan sebagai 'housing_clean.csv'
--- Validasi Akhir ---
Total Missing Values: 0
Total Duplikat: 0
Shape Akhir: (130, 7)
